# 2.2 Reshape

**Core series, class 4.** Our clean measurements have **many rows per person**: every blood pressure reading, every BMI. Most analyses need **one row per person**, with one column per variable. Changing the shape of a table is called **reshaping**. In this class we choose **one value per person** for each measure, and turn tables from **long** to **wide** (and back).

### By the end of this notebook you'll be able to

1. Tell **long** from **wide** data, and say when each is useful
2. Choose **one value per person** from many (latest, first, median…), and explain the choice
3. Turn long data into wide with `pivot_wider()`, and wide into long with `pivot_longer()`
4. Recognise and fix the most common `pivot_wider()` problem: values that aren't unique

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | How do we get from many measurements per person to one row per person? |
| 🧪 **The analysis** | Summaries per person (`group_by()` + `slice_max()`), then `pivot_wider()`; `pivot_longer()` for plotting |
| 🎯 **Why** | Models and tables need one row per person. **Which** value we keep (the latest, the first, the median) is a research decision that changes the results |
| ✅ **What we conclude** | One row per person with BMI, SBP, HbA1c, and LDL, and one with the two PTSD answers |
| ⚠️ **What it can't tell us** | Whether the value we kept was measured at a **meaningful time** (e.g. before the diagnosis) |
| 🔗 **Connections** | Uses the clean files from **2.1**; **2.3** joins the wide tables into one analysis file |

| | |
|---|---|
| 📂 **Uses** | `cohort_clean.csv`, `survey_clean.csv`, `measurement_clean.csv` (from **2.1**) |
| 💾 **Saves** | `survey_wide.csv`, `measurements_wide.csv` |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts: long and wide | 10 |
| 2. Load | 5 |
| 3. Survey answers: long to wide | 10 |
| 4. Measurements: one value per person | 20 |
| 5. Measurements: long to wide | 10 |
| 6. Wide to long, for a plot | 10 |
| Wrap-up | 5 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 70 minutes |
| ⏱️ **Run time** | about **1 minute** to run every cell |
| 💰 **Data cost** | **$0**: no BigQuery queries |
| 💰 **App cost** | about **$0.27 an hour** while the app is running. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| **Estimated total** | about **$0.35** per student for this class (app time only) |

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts: long and wide

The **same data** can be stored in two shapes:

**Long:** one row per **observation** (here: per person per measure)

| person_id | measure | value |
|---|---|---|
| A | BMI | 31.2 |
| A | SBP | 128 |
| B | BMI | 24.5 |

**Wide:** one row per **person**, one column per measure

| person_id | BMI | SBP |
|---|---|---|
| A | 31.2 | 128 |
| B | 24.5 | `NA` |

| Shape | Good for |
|---|---|
| **Long** | storing many observations; filtering and summarising by group; **plots** with ggplot2 (one column for the value, one for the group) |
| **Wide** | **tables** and **models**: one row per person, each variable a column |

Neither is "correct": you reshape to suit the next step.

| Function | Turns | Key arguments |
|---|---|---|
| `pivot_wider()` | long → wide | `names_from` (the column whose values become new column names), `values_from` (the column with the values) |
| `pivot_longer()` | wide → long | `cols` (the columns to stack), `names_to`, `values_to` |

## Part 2. Load

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

cohort_clean      <- load_data("cohort_clean.csv",      col_types = cols(person_id = "c"))
survey_clean      <- load_data("survey_clean.csv",      col_types = cols(person_id = "c"))
measurement_clean <- load_data("measurement_clean.csv", col_types = cols(person_id = "c"))

head(survey_clean %>% select(-person_id), 3)        # long: one row per person per question
head(measurement_clean %>% select(-person_id), 3)   # long: one row per measurement

**🔍 Read the output:** both tables are **long**. (We hide `person_id` when printing rows: it's an identifier.)

## Part 3. Survey answers: long to wide

Each person has at most one answer per question (2.1 kept the latest), so `pivot_wider()` can turn the `question` values into **column names**, filled with the `answer`.

In [ ]:
survey_wide <- survey_clean %>%
  pivot_wider(id_cols = person_id, names_from = question, values_from = answer)

nrow(survey_wide)                              # people with at least one answer
n_distinct(survey_wide$person_id)              # should be the same: one row per person
survey_wide %>% count(ptsd_doctor, ptsd_treatment) %>% hide_small_counts()

save_data(survey_wide, "survey_wide.csv")

**🔍 Read the output:** one row per person who answered, with a `ptsd_doctor` and a `ptsd_treatment` column. The cross-table shows how the answers combine; `NA` means that person answered one question but not the other.

**Always check** that a wide table has **one row per person**: `nrow()` equals `n_distinct(person_id)`.

## Part 4. Measurements: one value per person

### How many measurements per person?

In [ ]:
per_person <- measurement_clean %>% count(person_id, measure)

per_person %>%
  group_by(measure) %>%
  summarise(people = n(), median_per_person = median(n),
            max_per_person = max(n))   # a maximum describes ONE person: look, but don't share it

**🔍 Read the output:** the typical person has **many** blood pressure readings (a median of 46 in our test run), and some people have **thousands**, likely from monitoring during hospital stays. HbA1c and LDL are measured less often (a median of 4 to 5 per person).

### Which value?

With many values per person, we must choose. There's no single right answer:

| Choice | Code idea | Good for | Watch out for |
|---|---|---|---|
| **Latest** | `slice_max(measurement_date)` | "current" status | for cases, often **after** diagnosis and treatment |
| **First** | `slice_min(measurement_date)` | the earliest picture | old measurements, different eras |
| **Median** | `summarise(median(value))` | a stable "typical" value | mixes before and after events |
| **Closest before a date** | filter dates before an index date, then latest | **baseline** before an exposure or diagnosis | needs a date for everyone |

**Decision for this series:** the **latest** value, because it's simple and every person has one. Its weakness matters: for cases, the latest HbA1c or LDL was usually measured **after** diabetes was diagnosed and treated (e.g. with statins, which lower LDL). We'll see that in 3.2.

`slice_max(measurement_date, n = 1, with_ties = FALSE)` keeps the row with the latest date in each group; `with_ties = FALSE` keeps just one if two share the date.

In [ ]:
measurement_latest <- measurement_clean %>%
  group_by(person_id, measure) %>%
  slice_max(measurement_date, n = 1, with_ties = FALSE) %>%
  ungroup()

c(before = nrow(measurement_clean), after = nrow(measurement_latest))
measurement_latest %>% count(measure)

**🔍 Read the output:** from about a million rows to at most 4 per person: one per measure.

## Part 5. Measurements: long to wide

In [ ]:
measurements_wide <- measurement_latest %>%
  pivot_wider(id_cols = person_id, names_from = measure, values_from = value_as_number)

nrow(measurements_wide) == n_distinct(measurements_wide$person_id)   # one row per person?
summary(measurements_wide %>% select(-person_id))

save_data(measurements_wide, "measurements_wide.csv")

**🔍 Read the output:** `TRUE` (one row per person), then each measure's summary. `NA's` counts people without that measure: they'll keep `NA` when we join in 2.3.

### ✍️ Your turn

What happens if you **skip** the one-value-per-person step and pivot `measurement_clean` directly? Try it on a few people.

<details>
<summary><b>Show answer</b></summary>

```r
measurement_clean %>%
  filter(person_id %in% head(unique(person_id), 5)) %>%
  pivot_wider(id_cols = person_id, names_from = measure, values_from = value_as_number)
```

R warns that **values are not uniquely identified**, and each cell becomes a **list** of all that person's values (`<dbl [37]>`). It's the most common `pivot_wider()` problem. The fix is to decide **which** value you want first (as we did with `slice_max()`), or to tell `pivot_wider()` how to combine them, e.g. `values_fn = median`.

</details>

In [ ]:
# Your code here

## Part 6. Wide to long, for a plot

ggplot2 likes **long** data: to draw one histogram per measure, it needs the values in **one** column and the measure name in another. `pivot_longer()` turns the wide table back.

We also need each person's group (case or control), so we add it with a join (the topic of 2.3).

In [ ]:
plot_data <- measurements_wide %>%
  pivot_longer(cols = c(BMI, SBP, HbA1c, LDL), names_to = "measure", values_to = "value",
               values_drop_na = TRUE) %>%
  left_join(cohort_clean %>% select(person_id, t2d), by = "person_id")

head(plot_data %>% select(-person_id), 4)

ggplot(plot_data, aes(value, fill = t2d)) +
  geom_histogram(bins = 40, alpha = 0.6, position = "identity") +
  facet_wrap(~ measure, scales = "free") +
  scale_fill_manual(values = c(No = "#3182bd", Yes = "#d95f02")) +
  theme_bw() +
  labs(x = "Latest value", y = "People", fill = "Type 2 diabetes",
       title = "Latest measurements, cases vs controls")

**🔍 Read the output:**

- `pivot_longer()` made one row per person per measure again. `values_drop_na = TRUE` dropped the `NA`s.
- `facet_wrap(~ measure, scales = "free")` draws one panel per measure, each with its own axis.
- **HbA1c** separates the groups most clearly: it's one of the tests used to **diagnose** diabetes, so it can't be used to "predict" it (more in 3.2).
- **BMI** is higher in cases; **LDL** may be **lower** in cases: many people with diabetes take statins.

## ⚠️ Common errors

| Message | Means | Fix |
|---|---|---|
| *Values from `value_as_number` are not uniquely identified* | More than one value per person per measure | Choose one value first (`slice_max()`), or use `values_fn` |
| Cells show `<dbl [n]>` or `NULL` | Same as above: list-columns | Same fix |
| *Can't combine … `<character>` and `<double>`* in `pivot_longer()` | The columns being stacked have different types | Stack only columns of the same type, or convert first |

## ✅ Key takeaways

- **Long** for storing, summarising, and plotting; **wide** for tables and models
- Choosing **one value per person** is a research decision: say which and why
- After `pivot_wider()`, check **one row per person**
- `pivot_longer()` turns wide data back into long, e.g. for ggplot2

**Next:** **2.3 join_and_bind**: combine the cohort, measurements, survey answers, and CKD dates into one analysis table.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).